In [1]:
# ════════════════════════════════════════════════════════════════════════════════════
# PBT-V v3: Making E+M NECESSARY — Tasks That Module V Cannot Solve Alone
# Predictive Boundary Theory — Ninthanawat N.
# Boundary Research Initiative, Bangkok, Thailand
#
# MOTIVATION (from Compact State):
#   Vision v2 achieved 99.8% but Ablation = 0.0pp (ceiling effect)
#   E+M "woke up" (Pi movement 300x, gamma 50-250x) but V handled everything alone
#   => Need tasks where V CANNOT solve without E+M
#
# NEW TASK TYPES:
#   1. Gradual Drift Anomaly:
#      - Each consecutive frame differs by only ~14% (small epsilon)
#      - V sees each frame and says "looks ok-ish"
#      - But cumulative drift is 100% (car → airplane)
#      - Only M (temporal accumulation) can detect the drift
#
#   2. Context-Dependent Anomaly:
#      - Same object (frog) appears in both sequences
#      - Road context → frog = ANOMALY (frog on road)
#      - Nature context → frog = SAFE (frog in nature)
#      - V sees "frog" identically → cannot distinguish
#      - Only M (remembers prior context) can classify correctly
#
# Model: DINOv2-Base (768 dim, 12 layers)
# Dataset: CIFAR-10 Temporal Sequences (enhanced with drift + context tasks)
# Requirements: Colab Pro with A100 GPU
# ════════════════════════════════════════════════════════════════════════════════════

# ════════════════════════════════════════════
# CELL 0: Install
# ════════════════════════════════════════════
!pip install -q transformers accelerate datasets scikit-learn matplotlib seaborn torchvision



In [4]:
# ════════════════════════════════════════════
# CELL 1: Architecture — PBT-V v3 (DINOv2-Base)
# ════════════════════════════════════════════
import os, json, random, math, warnings, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from transformers import AutoImageProcessor, AutoModel
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torchvision
import torchvision.transforms as transforms
from PIL import Image
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB)')

print("\n" + "=" * 90)
print("  PBT-V v3: Making E+M NECESSARY")
print("  Tasks That Module V Cannot Solve Alone")
print("  DINOv2-Base | CIFAR-10 Temporal Sequences")
print("  Predictive Boundary Theory — Ninthanawat N.")
print("=" * 90)

# ─── Load DINOv2-Base ───
MODEL_ID = "facebook/dinov2-base"
print(f"\n Loading {MODEL_ID}...")
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
base_model = AutoModel.from_pretrained(MODEL_ID).to(device)
base_model.eval()
for p in base_model.parameters():
    p.requires_grad_(False)

D_MODEL = 768      # DINOv2-Base hidden dim
N_LAYERS = 12      # 12 transformer layers
NUM_HEADS = 12     # 12 attention heads
D_K = D_MODEL // NUM_HEADS  # 64

print(f"  DINOv2-Base loaded")
print(f"\n  PBT Architecture Mapping:")
print(f"  C (d_model) = {D_MODEL}")
print(f"  sigma (heads)   = {NUM_HEADS}")
print(f"  rho (d_k)     = {D_K}")
print(f"  tau (layers)  = {N_LAYERS}")
print(f"  sigma x rho = {NUM_HEADS} x {D_K} = {NUM_HEADS * D_K} {'= C' if NUM_HEADS*D_K==D_MODEL else ''}")

# ════════════════════════════════════════════
# PBT-V ADAPTER v3
# Same architecture as v2 — the challenge is in the TASK, not the model
# ════════════════════════════════════════════
class PBTVisionAdapterV3(nn.Module):
    """
    PBT-V v3: Full E->V->M pipeline for temporal vision processing.
    Architecture identical to v2 — proving that E+M modules are
    NECESSARY for harder tasks, not just "nice to have".

    Module E: epsilon_l(t) = Pi_l * (h_t(l) - h_{t-1}(l))   [temporal prediction error]
    Module V: valence_probes(epsilon_l) -> [V_pain, V_pleasure, V_epistemic]
    Module M: V_acc(t) = v_step(t) + (1-gamma) * V_acc(t-1)  [temporal EMA]

    KEY DIFFERENCE from v2:
    - Classifier receives h_last + v_feat + v_acc (same as v2)
    - But now v_acc carries CRITICAL information that h_last alone cannot provide:
      * For gradual drift: v_acc tracks cumulative epsilon over time
      * For context-dependent: v_acc encodes prior context
    """
    def __init__(self, d_model=D_MODEL, n_layers=N_LAYERS):
        super().__init__()
        self.d_model = d_model
        self.n_layers = n_layers

        # Module E: learnable precision per layer (log-space for positive guarantee)
        self.log_precision = nn.Parameter(torch.zeros(n_layers))

        # Module V: classify prediction error -> 3D valence
        self.valence_probes = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, 256), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(256, 3), nn.Sigmoid()
            ) for _ in range(n_layers)
        ])

        # Module M: differential decay (gamma_p < gamma_pl < gamma_e enforced via sort)
        self.raw_gamma = nn.Parameter(torch.tensor([-2.44, -1.92, -1.27]))

        # Classifier: [h_last | v_feat | v_acc] -> safe/unsafe
        in_features = d_model + (n_layers * 3) + 3
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 2)
        )

        # Internal state
        self.prev_h = None
        self.v_acc = None

    def get_precision(self):
        return torch.exp(self.log_precision)  # always positive

    def get_gamma(self):
        return torch.sigmoid(self.raw_gamma).sort().values  # enforced ordering

    def reset_state(self):
        self.prev_h = None
        self.v_acc = None

    def forward(self, h_current, return_valence=False):
        """
        h_current: [B, N_LAYERS, D_MODEL] - hidden states from current frame
        """
        B, L, D = h_current.shape

        if self.prev_h is None or self.prev_h.shape[0] != B:
            self.prev_h = h_current.detach()
            self.v_acc = torch.zeros(B, 3, device=h_current.device)

        # Module E: temporal prediction error
        precision = self.get_precision().view(1, -1, 1)  # [1, L, 1]
        epsilon = (h_current - self.prev_h) * precision   # [B, L, D]

        # Module V: classify each epsilon_l
        valences = []
        for l in range(self.n_layers):
            v_l = self.valence_probes[l](epsilon[:, l, :])  # [B, 3]
            valences.append(v_l)
        v_feat = torch.cat(valences, dim=1)        # [B, L*3]
        v_layers = torch.stack(valences, dim=1)    # [B, L, 3]
        v_step = v_layers.mean(dim=1)              # [B, 3]

        # Module M: temporal accumulation with differential decay
        gamma = self.get_gamma()
        self.v_acc = v_step + (1 - gamma) * self.v_acc

        # Classifier
        h_last = h_current[:, -1, :]  # [B, D]
        combined = torch.cat([h_last, v_feat, self.v_acc], dim=1)
        logits = self.classifier(combined)

        # Update prediction for next frame
        self.prev_h = h_current.detach()

        if return_valence:
            return logits, self.v_acc.detach(), v_layers.detach(), epsilon.detach()
        return logits, self.v_acc.detach(), v_feat.detach()

adapter = PBTVisionAdapterV3().to(device)
total_params = sum(p.numel() for p in adapter.parameters())
print(f"\n  PBT-V v3 Adapter: {total_params:,} trainable params")
print(f"  Module E: {sum(p.numel() for p in [adapter.log_precision]):,} (precision)")
print(f"  Module V: {sum(p.numel() for p in adapter.valence_probes.parameters()):,} (probes)")
print(f"  Module M: {sum(p.numel() for p in [adapter.raw_gamma]):,} (gamma)")
print(f"  Classifier: {sum(p.numel() for p in adapter.classifier.parameters()):,}")



Device: cuda
GPU: NVIDIA A100-SXM4-40GB (39 GB)

  PBT-V v3: Making E+M NECESSARY
  Tasks That Module V Cannot Solve Alone
  DINOv2-Base | CIFAR-10 Temporal Sequences
  Predictive Boundary Theory — Ninthanawat N.

 Loading facebook/dinov2-base...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  DINOv2-Base loaded

  PBT Architecture Mapping:
  C (d_model) = 768
  sigma (heads)   = 12
  rho (d_k)     = 64
  tau (layers)  = 12
  sigma x rho = 12 x 64 = 768 = C

  PBT-V v3 Adapter: 2,851,253 trainable params
  Module E: 12 (precision)
  Module V: 2,371,620 (probes)
  Module M: 3 (gamma)
  Classifier: 479,618


In [5]:
# ════════════════════════════════════════════
# CELL 2: Create Enhanced CIFAR-10 Temporal Sequences
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  Creating Enhanced CIFAR-10 Sequences")
print("  NEW: Gradual Drift + Context-Dependent Anomaly")
print("=" * 90)

# Download CIFAR-10
print("  Downloading CIFAR-10...")
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)

# CIFAR-10 classes:
# 0=airplane, 1=automobile, 2=bird, 3=cat, 4=deer,
# 5=dog, 6=frog, 7=horse, 8=ship, 9=truck
#
# PBT-V scenario: "robot driving on road"
#   Normal context (road): automobile(1), truck(9)
#   Anomaly: airplane(0), ship(8) = wrong on road
#   Nature context: bird(2), deer(4), dog(5), horse(7) = natural environment
#   Context-test object: frog(6) = same object, different meaning by context
NORMAL_ROAD = [1, 9]           # automobile, truck
ANOMALY_ROAD = [0, 8]          # airplane, ship (clearly wrong on road)
NATURE_CLASSES = [2, 4, 5, 7]  # bird, deer, dog, horse
CONTEXT_TEST_CLASS = 6         # frog: ANOMALY on road, SAFE in nature

# Sort images by class
images_by_class = {i: [] for i in range(10)}
for img, label in cifar_train:
    images_by_class[label].append(img)
for img, label in cifar_test:
    images_by_class[label].append(img)

print(f"  Images per class: { {k: len(v) for k, v in images_by_class.items()} }")

SEQ_LEN = 8  # 8 frames per sequence

# ─── Image Blending for Gradual Drift ───
def blend_images(img1, img2, alpha):
    """
    Blend two PIL images: result = (1-alpha)*img1 + alpha*img2
    alpha=0 -> pure img1, alpha=1 -> pure img2
    """
    arr1 = np.array(img1).astype(np.float32)
    arr2 = np.array(img2).astype(np.float32)
    blended = (1.0 - alpha) * arr1 + alpha * arr2
    return Image.fromarray(np.clip(blended, 0, 255).astype(np.uint8))

# ─── Sequence Generators ───

def create_normal_sequence(context="road", seq_len=SEQ_LEN):
    """
    TYPE 1: Normal — consistent context, no anomaly
    Road: 8 frames of automobile/truck
    Nature: 8 frames of bird/deer/dog/horse
    All labels = 0 (SAFE)
    """
    classes = NORMAL_ROAD if context == "road" else NATURE_CLASSES
    frames, labels = [], []
    for _ in range(seq_len):
        cls = random.choice(classes)
        frames.append(random.choice(images_by_class[cls]))
        labels.append(0)
    return frames, labels

def create_sudden_anomaly(seq_len=SEQ_LEN):
    """
    TYPE 2: Sudden Anomaly (CONTROL — V should handle this)
    4 normal road frames + 4 clearly anomalous frames
    = Big epsilon at transition point, V sees it immediately
    """
    frames, labels = [], []
    for _ in range(seq_len // 2):
        cls = random.choice(NORMAL_ROAD)
        frames.append(random.choice(images_by_class[cls]))
        labels.append(0)
    for _ in range(seq_len - seq_len // 2):
        cls = random.choice(ANOMALY_ROAD)
        frames.append(random.choice(images_by_class[cls]))
        labels.append(1)
    return frames, labels

def create_gradual_drift(seq_len=SEQ_LEN):
    """
    TYPE 3: Gradual Drift Anomaly (E+M CRITICAL)

    Alpha-blend from normal to anomalous over 8 frames:
      Frame 0: alpha=0.00 (100% car)         -> SAFE
      Frame 1: alpha=0.14 (86% car, 14% plane) -> SAFE
      Frame 2: alpha=0.29 (71% car, 29% plane) -> SAFE
      Frame 3: alpha=0.43 (57% car, 43% plane) -> SAFE
      Frame 4: alpha=0.57 (43% car, 57% plane) -> ANOMALY  <-- drift accumulated!
      Frame 5: alpha=0.71 (29% car, 71% plane) -> ANOMALY
      Frame 6: alpha=0.86 (14% car, 86% plane) -> ANOMALY
      Frame 7: alpha=1.00 (100% plane)         -> ANOMALY

    KEY: |epsilon| per step = alpha_delta = 1/7 ~ 0.14 (SMALL, V can't see)
         |epsilon_total| = 1.0 (LARGE, M accumulates and detects)

    WHY V FAILS:
    - Frame 3 (alpha=0.43) and Frame 4 (alpha=0.57) differ by only 14%
    - Their hidden states h_3 and h_4 are very similar
    - V classifies based on h_last alone -> ambiguous at the boundary
    - M has accumulated 4 frames of consistent drift -> clear signal
    """
    # Pick source (normal) and target (anomaly)
    src_cls = random.choice(NORMAL_ROAD)
    tgt_cls = random.choice(ANOMALY_ROAD)
    src_img = random.choice(images_by_class[src_cls])
    tgt_img = random.choice(images_by_class[tgt_cls])

    frames, labels = [], []
    for i in range(seq_len):
        alpha = i / (seq_len - 1)  # 0.0 to 1.0
        blended = blend_images(src_img, tgt_img, alpha)
        frames.append(blended)
        # SAFE while alpha < 0.5, ANOMALY when drift has accumulated enough
        labels.append(0 if alpha < 0.5 else 1)

    return frames, labels

def create_gradual_drift_safe(seq_len=SEQ_LEN):
    """
    TYPE 3b: Gradual Drift but SAFE (control for drift)

    Drift between two NORMAL classes (automobile <-> truck)
    = Same blending pattern, but drift stays within safe territory
    = Teaches model that "drift alone is not bad, drift INTO anomaly is bad"
    """
    src_cls = 1   # automobile
    tgt_cls = 9   # truck (both normal road objects)
    src_img = random.choice(images_by_class[src_cls])
    tgt_img = random.choice(images_by_class[tgt_cls])

    frames, labels = [], []
    for i in range(seq_len):
        alpha = i / (seq_len - 1)
        blended = blend_images(src_img, tgt_img, alpha)
        frames.append(blended)
        labels.append(0)  # always SAFE — drift within normal range

    return frames, labels

def create_context_road_frog(seq_len=SEQ_LEN):
    """
    TYPE 4a: Context-Dependent — ROAD context + frog = ANOMALY

    Frames 0-3: automobile/truck (road context)
    Frames 4-7: frog (ANOMALY — frog does not belong on road)

    WHY V FAILS:
    - V sees "frog" at frame 4 — same hidden state as in nature context
    - V cannot know this frog is anomalous without knowing the prior context
    - M accumulated "road, road, road, road" -> knows frog is out of place
    """
    frames, labels = [], []
    # Road context
    for _ in range(seq_len // 2):
        cls = random.choice(NORMAL_ROAD)
        frames.append(random.choice(images_by_class[cls]))
        labels.append(0)
    # Frog in road context = ANOMALY
    for _ in range(seq_len - seq_len // 2):
        frames.append(random.choice(images_by_class[CONTEXT_TEST_CLASS]))
        labels.append(1)  # ANOMALY: frog on road
    return frames, labels

def create_context_nature_frog(seq_len=SEQ_LEN):
    """
    TYPE 4b: Context-Dependent — NATURE context + frog = SAFE

    Frames 0-3: bird/deer/dog/horse (nature context)
    Frames 4-7: frog (SAFE — frog belongs in nature)

    KEY CONTRAST with Type 4a:
    - The frog frames are from the SAME class (class 6)
    - V sees identical "frog" hidden states in both 4a and 4b
    - Only M distinguishes: accumulated "nature" context -> frog is normal
    """
    frames, labels = [], []
    # Nature context
    for _ in range(seq_len // 2):
        cls = random.choice(NATURE_CLASSES)
        frames.append(random.choice(images_by_class[cls]))
        labels.append(0)
    # Frog in nature context = SAFE
    for _ in range(seq_len - seq_len // 2):
        frames.append(random.choice(images_by_class[CONTEXT_TEST_CLASS]))
        labels.append(0)  # SAFE: frog in nature
    return frames, labels

# ─── Generate Dataset ───
N_NORMAL_ROAD = 300
N_NORMAL_NATURE = 200
N_SUDDEN = 200       # Control: V can handle
N_DRIFT_ANOM = 300   # E+M critical
N_DRIFT_SAFE = 150   # Control for drift
N_CTX_ROAD = 200     # Frog on road = ANOMALY
N_CTX_NATURE = 200   # Frog in nature = SAFE

print(f"\n  Generating sequences...")
print(f"    Normal (road):       {N_NORMAL_ROAD}")
print(f"    Normal (nature):     {N_NORMAL_NATURE}")
print(f"    Sudden anomaly:      {N_SUDDEN}  (control - V can handle)")
print(f"    Gradual drift anom:  {N_DRIFT_ANOM}  (E+M CRITICAL)")
print(f"    Gradual drift safe:  {N_DRIFT_SAFE}  (drift control)")
print(f"    Context road+frog:   {N_CTX_ROAD}  (E+M CRITICAL)")
print(f"    Context nature+frog: {N_CTX_NATURE}  (E+M CRITICAL)")

all_sequences = []

for _ in range(N_NORMAL_ROAD):
    frames, labels = create_normal_sequence("road")
    all_sequences.append({"frames": frames, "labels": labels, "type": "normal_road"})

for _ in range(N_NORMAL_NATURE):
    frames, labels = create_normal_sequence("nature")
    all_sequences.append({"frames": frames, "labels": labels, "type": "normal_nature"})

for _ in range(N_SUDDEN):
    frames, labels = create_sudden_anomaly()
    all_sequences.append({"frames": frames, "labels": labels, "type": "sudden_anomaly"})

for _ in range(N_DRIFT_ANOM):
    frames, labels = create_gradual_drift()
    all_sequences.append({"frames": frames, "labels": labels, "type": "gradual_drift"})

for _ in range(N_DRIFT_SAFE):
    frames, labels = create_gradual_drift_safe()
    all_sequences.append({"frames": frames, "labels": labels, "type": "drift_safe"})

for _ in range(N_CTX_ROAD):
    frames, labels = create_context_road_frog()
    all_sequences.append({"frames": frames, "labels": labels, "type": "context_road_frog"})

for _ in range(N_CTX_NATURE):
    frames, labels = create_context_nature_frog()
    all_sequences.append({"frames": frames, "labels": labels, "type": "context_nature_frog"})

random.shuffle(all_sequences)

# Split: 70% train, 15% val, 15% test
n_total = len(all_sequences)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
train_seqs = all_sequences[:n_train]
val_seqs = all_sequences[n_train:n_train+n_val]
test_seqs = all_sequences[n_train+n_val:]

print(f"\n  Total: {n_total} sequences x {SEQ_LEN} frames = {n_total * SEQ_LEN} frames")
print(f"  Train: {len(train_seqs)} | Val: {len(val_seqs)} | Test: {len(test_seqs)}")

# Count types in each split
for split_name, split_data in [("Train", train_seqs), ("Val", val_seqs), ("Test", test_seqs)]:
    type_counts = {}
    for s in split_data:
        type_counts[s["type"]] = type_counts.get(s["type"], 0) + 1
    print(f"  {split_name}: {type_counts}")




  Creating Enhanced CIFAR-10 Sequences
  NEW: Gradual Drift + Context-Dependent Anomaly


100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


  Images per class: {0: 6000, 1: 6000, 2: 6000, 3: 6000, 4: 6000, 5: 6000, 6: 6000, 7: 6000, 8: 6000, 9: 6000}

  Generating sequences...
    Normal (road):       300
    Normal (nature):     200
    Sudden anomaly:      200  (control - V can handle)
    Gradual drift anom:  300  (E+M CRITICAL)
    Gradual drift safe:  150  (drift control)
    Context road+frog:   200  (E+M CRITICAL)
    Context nature+frog: 200  (E+M CRITICAL)

  Total: 1550 sequences x 8 frames = 12400 frames
  Train: 1085 | Val: 232 | Test: 233
  Train: {'gradual_drift': 222, 'normal_road': 206, 'normal_nature': 132, 'sudden_anomaly': 141, 'context_road_frog': 142, 'context_nature_frog': 147, 'drift_safe': 95}
  Val: {'drift_safe': 32, 'context_road_frog': 27, 'normal_road': 47, 'context_nature_frog': 24, 'sudden_anomaly': 34, 'gradual_drift': 39, 'normal_nature': 29}
  Test: {'drift_safe': 23, 'normal_nature': 39, 'normal_road': 47, 'context_road_frog': 31, 'gradual_drift': 39, 'sudden_anomaly': 25, 'context_nature

In [6]:
# ════════════════════════════════════════════
# CELL 3: Pre-compute Hidden States
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  Pre-computing Hidden States (DINOv2-Base)")
print("=" * 90)

def extract_sequence_features(sequences, desc=""):
    """Extract DINOv2 CLS tokens for all frames in all sequences."""
    all_h = []      # list of [SEQ_LEN, N_LAYERS, D_MODEL]
    all_labels = []  # list of [SEQ_LEN]
    all_types = []

    t0 = time.time()
    for i, seq in enumerate(sequences):
        seq_h = []
        for frame_img in seq["frames"]:
            inputs = processor(images=frame_img, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = base_model(**inputs, output_hidden_states=True)
            # CLS token from each layer
            layer_h = torch.stack([
                outputs.hidden_states[l+1][:, 0, :].squeeze(0).cpu()
                for l in range(N_LAYERS)
            ], dim=0)  # [N_LAYERS, D_MODEL]
            seq_h.append(layer_h)

        all_h.append(torch.stack(seq_h, dim=0))  # [SEQ_LEN, N_LAYERS, D_MODEL]
        all_labels.append(torch.tensor(seq["labels"]))
        all_types.append(seq["type"])

        if (i+1) % 200 == 0:
            elapsed = time.time() - t0
            print(f"    {desc} {i+1}/{len(sequences)} ({elapsed:.0f}s)")

    elapsed = time.time() - t0
    print(f"  {desc} done: {len(sequences)} sequences in {elapsed:.0f}s")
    return all_h, all_labels, all_types

print("\n  Pre-computing TRAIN...")
train_h, train_labels, train_types = extract_sequence_features(train_seqs, "Train")
print("\n  Pre-computing VAL...")
val_h, val_labels, val_types = extract_sequence_features(val_seqs, "Val")
print("\n  Pre-computing TEST...")
test_h, test_labels, test_types = extract_sequence_features(test_seqs, "Test")




  Pre-computing Hidden States (DINOv2-Base)

  Pre-computing TRAIN...
    Train 200/1085 (17s)
    Train 400/1085 (33s)
    Train 600/1085 (49s)
    Train 800/1085 (65s)
    Train 1000/1085 (81s)
  Train done: 1085 sequences in 88s

  Pre-computing VAL...
    Val 200/232 (16s)
  Val done: 232 sequences in 19s

  Pre-computing TEST...
    Test 200/233 (16s)
  Test done: 233 sequences in 19s


In [7]:
# ════════════════════════════════════════════
# CELL 4: Training with BPTT
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  Training PBT-V v3 (BPTT — E+M Must Learn to Matter)")
print("=" * 90)

# Differential LR: Module E+M get higher LR to encourage learning
optimizer = AdamW([
    {'params': adapter.classifier.parameters(), 'lr': 1e-3},
    {'params': adapter.valence_probes.parameters(), 'lr': 1e-3},
    {'params': [adapter.log_precision], 'lr': 5e-3, 'weight_decay': 0},
    {'params': [adapter.raw_gamma], 'lr': 5e-3, 'weight_decay': 0},
])
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

EPOCHS = 40  # More epochs for harder task
best_val_acc = 0
best_state = None
history = {'loss': [], 'train_acc': [], 'val_acc': [],
           'gamma_p': [], 'gamma_pl': [], 'gamma_e': [],
           'precision_mean': [], 'precision_max': []}

adapter.train()
t0 = time.time()

for epoch in range(EPOCHS):
    epoch_loss = 0
    all_preds, all_true = [], []

    indices = list(range(len(train_h)))
    random.shuffle(indices)

    for idx in indices:
        seq_h = train_h[idx].to(device)           # [SEQ_LEN, N_LAYERS, D_MODEL]
        seq_labels = train_labels[idx].to(device)  # [SEQ_LEN]

        adapter.reset_state()
        optimizer.zero_grad()

        seq_loss = 0
        for t in range(SEQ_LEN):
            h_t = seq_h[t].unsqueeze(0)       # [1, N_LAYERS, D_MODEL]
            label_t = seq_labels[t].unsqueeze(0)

            logits, v_acc, v_feat = adapter(h_t)
            loss = criterion(logits, label_t)
            seq_loss = seq_loss + loss

            pred = logits.argmax(-1).item()
            all_preds.append(pred)
            all_true.append(label_t.item())

        seq_loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        optimizer.step()
        epoch_loss += seq_loss.item()

    scheduler.step()

    # Validation
    adapter.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for idx in range(len(val_h)):
            adapter.reset_state()
            for t in range(SEQ_LEN):
                h_t = val_h[idx][t].unsqueeze(0).to(device)
                logits, _, _ = adapter(h_t)
                val_preds.append(logits.argmax(-1).item())
                val_true.append(val_labels[idx][t].item())
    adapter.train()

    train_acc = accuracy_score(all_true, all_preds)
    val_acc = accuracy_score(val_true, val_preds)
    avg_loss = epoch_loss / len(train_h)

    gammas = adapter.get_gamma().detach().cpu().numpy()
    precs = adapter.get_precision().detach().cpu().numpy()

    history['loss'].append(avg_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['gamma_p'].append(gammas[0])
    history['gamma_pl'].append(gammas[1])
    history['gamma_e'].append(gammas[2])
    history['precision_mean'].append(precs.mean())
    history['precision_max'].append(precs.max())

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(adapter.state_dict())

    elapsed = time.time() - t0
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/{EPOCHS}: Loss={avg_loss:.4f} Train={train_acc:.1%} Val={val_acc:.1%} "
              f"gamma_p={gammas[0]:.4f} gamma_e={gammas[2]:.4f} Pi_max={precs.max():.4f} ({elapsed:.0f}s)")

adapter.load_state_dict(best_state)
adapter.eval()
print(f"\n  Best val accuracy: {best_val_acc:.1%}")

# Save
torch.save({"state_dict": adapter.state_dict(), "config": {
    "d_model": D_MODEL, "n_layers": N_LAYERS, "best_val_acc": best_val_acc,
    "n_train": len(train_h), "n_val": len(val_h), "n_test": len(test_h),
    "version": "v3", "task_types": ["normal_road", "normal_nature", "sudden_anomaly",
                                     "gradual_drift", "drift_safe",
                                     "context_road_frog", "context_nature_frog"]
}}, "pbt_vision_v3.pth")

# Plot training
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(history['loss'], 'b-', lw=2); axes[0,0].set_title('Loss'); axes[0,0].grid(True, alpha=0.3)
axes[0,1].plot(history['train_acc'], 'g-', lw=2, label='Train')
axes[0,1].plot(history['val_acc'], 'r-', lw=2, label='Val')
axes[0,1].legend(); axes[0,1].set_title('Accuracy'); axes[0,1].grid(True, alpha=0.3)
axes[1,0].plot(history['gamma_p'], 'r-', lw=2, label='gamma_pain')
axes[1,0].plot(history['gamma_pl'], 'g-', lw=2, label='gamma_pleasure')
axes[1,0].plot(history['gamma_e'], 'b-', lw=2, label='gamma_epistemic')
axes[1,0].legend(); axes[1,0].set_title('Module M: Gamma (Decay Rates)'); axes[1,0].grid(True, alpha=0.3)
axes[1,1].plot(history['precision_mean'], 'purple', lw=2, label='Pi mean')
axes[1,1].plot(history['precision_max'], 'orange', lw=2, label='Pi max')
axes[1,1].axhline(1.0, color='gray', ls='--'); axes[1,1].legend()
axes[1,1].set_title('Module E: Precision Pi_l'); axes[1,1].grid(True, alpha=0.3)
plt.suptitle(f'PBT-V v3 Training ({len(train_h)} sequences) — Ninthanawat N.', fontweight='bold')
plt.tight_layout(); plt.savefig('pbtv3_training.png', dpi=150); plt.show()




  Training PBT-V v3 (BPTT — E+M Must Learn to Matter)
  Epoch  5/40: Loss=1.4767 Train=95.9% Val=95.2% gamma_p=0.0044 gamma_e=0.0117 Pi_max=21.5530 (429s)
  Epoch 10/40: Loss=0.2705 Train=99.3% Val=95.9% gamma_p=0.0011 gamma_e=0.0033 Pi_max=45.4676 (856s)
  Epoch 15/40: Loss=0.1867 Train=99.6% Val=95.9% gamma_p=0.0006 gamma_e=0.0018 Pi_max=43.9003 (1284s)
  Epoch 20/40: Loss=0.0940 Train=99.9% Val=96.7% gamma_p=0.0004 gamma_e=0.0012 Pi_max=48.8368 (1712s)
  Epoch 25/40: Loss=0.0129 Train=100.0% Val=96.9% gamma_p=0.0003 gamma_e=0.0010 Pi_max=60.6432 (2141s)
  Epoch 30/40: Loss=0.0000 Train=100.0% Val=97.1% gamma_p=0.0003 gamma_e=0.0009 Pi_max=59.6696 (2569s)
  Epoch 35/40: Loss=0.0000 Train=100.0% Val=97.1% gamma_p=0.0003 gamma_e=0.0009 Pi_max=59.9301 (2997s)
  Epoch 40/40: Loss=0.0000 Train=100.0% Val=97.1% gamma_p=0.0003 gamma_e=0.0009 Pi_max=60.2530 (3425s)

  Best val accuracy: 97.2%


In [8]:
# ════════════════════════════════════════════
# CELL 5: Comprehensive Test Results
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  Comprehensive Test Results — PBT-V v3")
print("=" * 90)

adapter.eval()
test_preds, test_true, test_type_list = [], [], []

with torch.no_grad():
    for idx in range(len(test_h)):
        adapter.reset_state()
        for t in range(SEQ_LEN):
            h_t = test_h[idx][t].unsqueeze(0).to(device)
            logits, _, _ = adapter(h_t)
            test_preds.append(logits.argmax(-1).item())
            test_true.append(test_labels[idx][t].item())
            test_type_list.append(test_types[idx])

overall_acc = accuracy_score(test_true, test_preds)
print(f"\n  Overall Accuracy: {overall_acc:.1%} ({sum(1 for p,t in zip(test_preds,test_true) if p==t)}/{len(test_true)})")
print(f"\n{classification_report(test_true, test_preds, target_names=['SAFE','ANOMALY'], digits=4)}")

# Per-type breakdown
print("  By Sequence Type:")
type_accs = {}
for stype in ["normal_road", "normal_nature", "sudden_anomaly",
              "gradual_drift", "drift_safe",
              "context_road_frog", "context_nature_frog"]:
    idx = [i for i, t in enumerate(test_type_list) if t == stype]
    if idx:
        acc = accuracy_score([test_true[i] for i in idx], [test_preds[i] for i in idx])
        type_accs[stype] = acc
        marker = " <-- E+M CRITICAL" if stype in ["gradual_drift", "context_road_frog", "context_nature_frog"] else ""
        print(f"  {stype:25s}: {acc:.1%} ({len(idx)} frames){marker}")

# Confusion matrix
cm = confusion_matrix(test_true, test_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=["SAFE", "ANOMALY"], yticklabels=["SAFE", "ANOMALY"])
ax.set_title(f'PBT-V v3 — Overall Acc: {overall_acc:.1%} ({len(test_h)} sequences)')
plt.tight_layout(); plt.savefig('pbtv3_confusion.png', dpi=150); plt.show()




  Comprehensive Test Results — PBT-V v3

  Overall Accuracy: 97.2% (1812/1864)

              precision    recall  f1-score   support

        SAFE     0.9851    0.9798    0.9824      1484
     ANOMALY     0.9227    0.9421    0.9323       380

    accuracy                         0.9721      1864
   macro avg     0.9539    0.9609    0.9574      1864
weighted avg     0.9724    0.9721    0.9722      1864

  By Sequence Type:
  normal_road              : 99.7% (376 frames)
  normal_nature            : 100.0% (312 frames)
  sudden_anomaly           : 99.5% (200 frames)
  gradual_drift            : 94.9% (312 frames) <-- E+M CRITICAL
  drift_safe               : 95.1% (184 frames)
  context_road_frog        : 96.0% (248 frames) <-- E+M CRITICAL
  context_nature_frog      : 93.5% (232 frames) <-- E+M CRITICAL


In [9]:
# ════════════════════════════════════════════
# CELL 6: ABLATION BY TASK TYPE — THE KEY EXPERIMENT
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  ABLATION BY TASK TYPE: Proving E+M Are NECESSARY")
print("  (This is the key experiment of v3)")
print("=" * 90)

def evaluate_ablation_by_type(adapter_state, ablate_E=False, ablate_M=False, name=""):
    """
    Test with specific modules disabled, report per-type accuracy.
    Returns: {type: accuracy} dict
    """
    abl_adapter = PBTVisionAdapterV3().to(device)
    abl_adapter.load_state_dict(adapter_state)
    abl_adapter.eval()

    # Ablate Module E: set precision to 1.0 (neutral — no learned weighting)
    if ablate_E:
        with torch.no_grad():
            abl_adapter.log_precision.fill_(0.0)  # exp(0) = 1

    # Collect predictions per type
    type_preds = {}
    type_true = {}

    with torch.no_grad():
        for idx in range(len(test_h)):
            stype = test_types[idx]
            if stype not in type_preds:
                type_preds[stype] = []
                type_true[stype] = []

            abl_adapter.reset_state()

            for t in range(SEQ_LEN):
                h_t = test_h[idx][t].unsqueeze(0).to(device)

                if ablate_M:
                    # Disable temporal accumulation: reset v_acc each frame
                    abl_adapter.v_acc = torch.zeros(1, 3, device=device)

                logits, _, _ = abl_adapter(h_t)
                type_preds[stype].append(logits.argmax(-1).item())
                type_true[stype].append(test_labels[idx][t].item())

    results = {}
    for stype in type_preds:
        results[stype] = accuracy_score(type_true[stype], type_preds[stype])
    results["overall"] = accuracy_score(
        sum(type_true.values(), []),
        sum(type_preds.values(), [])
    )
    return results

trained_state = adapter.state_dict()

print("\n  Running ablations...")
res_full = evaluate_ablation_by_type(trained_state, name="Full (E+V+M)")
res_no_E = evaluate_ablation_by_type(trained_state, ablate_E=True, name="No E")
res_no_M = evaluate_ablation_by_type(trained_state, ablate_M=True, name="No M")
res_no_EM = evaluate_ablation_by_type(trained_state, ablate_E=True, ablate_M=True, name="No E+M")

# Print results table
print(f"\n{'='*90}")
print(f"  ABLATION RESULTS BY TASK TYPE")
print(f"{'='*90}")
print(f"\n  {'Task Type':<25s} {'Full E+V+M':>10s} {'No E':>10s} {'No M':>10s} {'V only':>10s} {'E+M Impact':>10s}")
print(f"  {'─'*75}")

all_types_sorted = ["normal_road", "normal_nature", "sudden_anomaly",
                    "gradual_drift", "drift_safe",
                    "context_road_frog", "context_nature_frog", "overall"]

for stype in all_types_sorted:
    if stype in res_full:
        full = res_full[stype]
        no_e = res_no_E.get(stype, 0)
        no_m = res_no_M.get(stype, 0)
        no_em = res_no_EM.get(stype, 0)
        impact = (full - no_em) * 100

        marker = ""
        if stype == "gradual_drift":
            marker = " ** DRIFT"
        elif stype in ["context_road_frog", "context_nature_frog"]:
            marker = " ** CTX"
        elif stype == "overall":
            marker = " <<"

        print(f"  {stype:<25s} {full:>9.1%} {no_e:>9.1%} {no_m:>9.1%} {no_em:>9.1%} {impact:>+9.1f}pp{marker}")

# Analysis
print(f"\n  === ANALYSIS ===")
drift_impact = (res_full.get("gradual_drift", 0) - res_no_EM.get("gradual_drift", 0)) * 100
ctx_road_impact = (res_full.get("context_road_frog", 0) - res_no_EM.get("context_road_frog", 0)) * 100
ctx_nature_impact = (res_full.get("context_nature_frog", 0) - res_no_EM.get("context_nature_frog", 0)) * 100
sudden_impact = (res_full.get("sudden_anomaly", 0) - res_no_EM.get("sudden_anomaly", 0)) * 100

print(f"  Sudden anomaly (control):  E+M impact = {sudden_impact:+.1f}pp  (V should handle this)")
print(f"  Gradual drift:             E+M impact = {drift_impact:+.1f}pp  (E+M should be critical)")
print(f"  Context road+frog:         E+M impact = {ctx_road_impact:+.1f}pp  (M should be critical)")
print(f"  Context nature+frog:       E+M impact = {ctx_nature_impact:+.1f}pp  (M should be critical)")

if drift_impact > 2.0 or ctx_road_impact > 2.0:
    print(f"\n  ======================================")
    print(f"  E+M ARE NECESSARY!")
    print(f"  Removing E+M drops accuracy significantly on:")
    if drift_impact > 2.0:
        print(f"    - Gradual drift: {drift_impact:+.1f}pp (V alone misses slow changes)")
    if ctx_road_impact > 2.0:
        print(f"    - Context-dependent: {ctx_road_impact:+.1f}pp / {ctx_nature_impact:+.1f}pp")
        print(f"      (V sees same object but M provides context)")
    print(f"  ======================================")
else:
    print(f"\n  E+M impact still small. Possible reasons:")
    print(f"    - DINOv2 hidden states may leak context information")
    print(f"    - Need even harder tasks (longer drift, more ambiguous objects)")

# Ablation bar chart — grouped by task type
fig, ax = plt.subplots(figsize=(16, 7))
task_types_plot = ["sudden_anomaly", "gradual_drift", "context_road_frog", "context_nature_frog"]
x = np.arange(len(task_types_plot))
width = 0.2

for i, (label, res, color) in enumerate([
    ("Full E+V+M", res_full, '#2ecc71'),
    ("No E", res_no_E, '#e67e22'),
    ("No M", res_no_M, '#3498db'),
    ("V only (No E+M)", res_no_EM, '#e74c3c'),
]):
    vals = [res.get(t, 0) for t in task_types_plot]
    bars = ax.bar(x + i*width, vals, width, label=label, color=color, edgecolor='black', lw=0.8)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.0%}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(["Sudden Anomaly\n(control)", "Gradual Drift\n(E+M critical)",
                     "Road+Frog\n(M critical)", "Nature+Frog\n(M critical)"])
ax.set_ylim(0, 1.15); ax.set_ylabel('Accuracy'); ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')
ax.set_title('PBT-V v3: Ablation by Task Type — Proving E+M Necessity\nNinthanawat N.', fontweight='bold')
plt.tight_layout(); plt.savefig('pbtv3_ablation_by_type.png', dpi=150); plt.show()




  ABLATION BY TASK TYPE: Proving E+M Are NECESSARY
  (This is the key experiment of v3)

  Running ablations...

  ABLATION RESULTS BY TASK TYPE

  Task Type                 Full E+V+M       No E       No M     V only E+M Impact
  ───────────────────────────────────────────────────────────────────────────
  normal_road                   99.7%    100.0%    100.0%    100.0%      -0.3pp
  normal_nature                100.0%    100.0%    100.0%    100.0%      +0.0pp
  sudden_anomaly                99.5%     99.5%     93.5%     94.0%      +5.5pp
  gradual_drift                 94.9%     92.9%     82.1%     78.5%     +16.3pp ** DRIFT
  drift_safe                    95.1%     97.8%    100.0%    100.0%      -4.9pp
  context_road_frog             96.0%     96.8%     54.0%     54.0%     +41.9pp ** CTX
  context_nature_frog           93.5%     91.8%    100.0%    100.0%      -6.5pp ** CTX
  overall                       97.2%     97.1%     90.2%     89.6%      +7.6pp <<

  === ANALYSIS ===
  Sudd

In [10]:
# ════════════════════════════════════════════
# CELL 7: Temporal EEG — Gradual Drift Sequence
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  Temporal EEG: Gradual Drift Anomaly (car -> airplane)")
print("=" * 90)

# Find a gradual drift sequence
drift_idx = next(i for i, t in enumerate(test_types) if t == "gradual_drift")

adapter.eval()
adapter.reset_state()

pain_drift, pleasure_drift, epistemic_drift, unsafe_drift = [], [], [], []
epsilon_norms_drift = []

with torch.no_grad():
    for t in range(SEQ_LEN):
        h_t = test_h[drift_idx][t].unsqueeze(0).to(device)
        logits, v_acc, v_layers, epsilon = adapter(h_t, return_valence=True)

        pain_drift.append(v_acc[0, 0].item())
        pleasure_drift.append(v_acc[0, 1].item())
        epistemic_drift.append(v_acc[0, 2].item())
        unsafe_drift.append(F.softmax(logits, dim=-1)[0, 1].item())
        epsilon_norms_drift.append(epsilon.norm(dim=-1).mean().item())

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Top: Valence accumulation
frames = list(range(SEQ_LEN))
ax1.plot(frames, pain_drift, 'r-o', lw=2, label='V_pain (Accumulated Threat)')
ax1.plot(frames, pleasure_drift, 'g-s', lw=2, label='V_pleasure (Accumulated Safety)')
ax1.plot(frames, epistemic_drift, 'b-^', lw=2, label='V_epistemic (Accumulated Surprise)')
ax1.axvspan(3.5, SEQ_LEN-0.5, alpha=0.1, color='red', label='Drift > 50% (ANOMALY)')
alphas = [f'{i/(SEQ_LEN-1):.0%}' for i in range(SEQ_LEN)]
ax1.set_xticks(frames)
ax1.set_xticklabels([f'F{i}\nalpha={alphas[i]}' for i in frames])
ax1.set_ylabel('Accumulated Valence (Module M)')
ax1.legend(loc='upper left'); ax1.grid(True, alpha=0.3)
ax1.set_title('Gradual Drift: Valence Accumulation Over Time', fontweight='bold')

# Bottom: epsilon per step + unsafe probability
ax2b = ax2.twinx()
ax2.bar(frames, epsilon_norms_drift, color='purple', alpha=0.5, label='|epsilon| per step (small!)')
ax2b.plot(frames, unsafe_drift, 'k-x', lw=2.5, ms=8, label='UNSAFE Prob')
ax2b.axhline(0.5, color='gray', ls='--', alpha=0.5)
ax2.set_xticks(frames)
ax2.set_xticklabels([f'F{i}\nalpha={alphas[i]}' for i in frames])
ax2.set_ylabel('|epsilon| (Prediction Error Per Step)')
ax2b.set_ylabel('UNSAFE Probability')
ax2b.set_ylim(-0.05, 1.05)

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2, loc='upper left')
ax2.set_title('Per-Step epsilon is SMALL — Only Accumulation Reveals Drift', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.suptitle('PBT-V v3: Gradual Drift Anomaly — Ninthanawat N.', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.savefig('pbtv3_eeg_drift.png', dpi=150); plt.show()

print(f"  epsilon norms per step: {[f'{e:.4f}' for e in epsilon_norms_drift]}")
print(f"  Max epsilon: {max(epsilon_norms_drift):.4f} (should be small per step)")
print(f"  UNSAFE prob trajectory: {[f'{u:.3f}' for u in unsafe_drift]}")




  Temporal EEG: Gradual Drift Anomaly (car -> airplane)
  epsilon norms per step: ['0.0000', '6.7731', '7.3627', '5.6330', '6.1419', '6.3971', '10.4237', '12.5598']
  Max epsilon: 12.5598 (should be small per step)
  UNSAFE prob trajectory: ['0.000', '0.000', '0.000', '0.000', '0.000', '0.000', '1.000', '1.000']


In [11]:
# ════════════════════════════════════════════
# CELL 8: Temporal EEG — Context-Dependent Comparison
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  Temporal EEG: Context-Dependent (Same Frog, Different Meaning)")
print("=" * 90)

# Find road+frog and nature+frog sequences
road_frog_idx = next(i for i, t in enumerate(test_types) if t == "context_road_frog")
nature_frog_idx = next(i for i, t in enumerate(test_types) if t == "context_nature_frog")

# Road context + frog
adapter.reset_state()
pain_road, unsafe_road, vacc_road = [], [], []
with torch.no_grad():
    for t in range(SEQ_LEN):
        h_t = test_h[road_frog_idx][t].unsqueeze(0).to(device)
        logits, v_acc, v_layers, epsilon = adapter(h_t, return_valence=True)
        pain_road.append(v_acc[0, 0].item())
        unsafe_road.append(F.softmax(logits, dim=-1)[0, 1].item())
        vacc_road.append(v_acc[0].clone().cpu().numpy())

# Nature context + frog
adapter.reset_state()
pain_nature, unsafe_nature, vacc_nature = [], [], []
with torch.no_grad():
    for t in range(SEQ_LEN):
        h_t = test_h[nature_frog_idx][t].unsqueeze(0).to(device)
        logits, v_acc, v_layers, epsilon = adapter(h_t, return_valence=True)
        pain_nature.append(v_acc[0, 0].item())
        unsafe_nature.append(F.softmax(logits, dim=-1)[0, 1].item())
        vacc_nature.append(v_acc[0].clone().cpu().numpy())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Road context
ax1.plot(frames, unsafe_road, 'r-o', lw=2.5, ms=8, label='UNSAFE Prob')
ax1.axvspan(3.5, SEQ_LEN-0.5, alpha=0.15, color='red', label='Frog Region')
ax1.axhline(0.5, color='gray', ls='--', alpha=0.5)
labels_road = ['Car']*4 + ['FROG']*4
ax1.set_xticks(frames)
ax1.set_xticklabels([f'F{i}\n{labels_road[i]}' for i in frames])
ax1.set_ylabel('UNSAFE Probability')
ax1.set_ylim(-0.05, 1.05)
ax1.set_title('Context A: ROAD + Frog = ANOMALY\n(M accumulated: "cars, cars, cars..." -> frog = wrong!)', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Right: Nature context
ax2.plot(frames, unsafe_nature, 'g-s', lw=2.5, ms=8, label='UNSAFE Prob')
ax2.axvspan(3.5, SEQ_LEN-0.5, alpha=0.15, color='green', label='Frog Region')
ax2.axhline(0.5, color='gray', ls='--', alpha=0.5)
labels_nature = ['Animal']*4 + ['FROG']*4
ax2.set_xticks(frames)
ax2.set_xticklabels([f'F{i}\n{labels_nature[i]}' for i in frames])
ax2.set_ylabel('UNSAFE Probability')
ax2.set_ylim(-0.05, 1.05)
ax2.set_title('Context B: NATURE + Frog = SAFE\n(M accumulated: "bird, deer, dog..." -> frog = fits!)', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('PBT-V v3: Same Object (Frog) — Different Context = Different Classification\n'
             'Module V sees identical frog, only Module M provides context — Ninthanawat N.',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.savefig('pbtv3_eeg_context.png', dpi=150); plt.show()

print(f"  Road+Frog UNSAFE probs:   {[f'{u:.3f}' for u in unsafe_road]}")
print(f"  Nature+Frog UNSAFE probs: {[f'{u:.3f}' for u in unsafe_nature]}")
print(f"  Frog frame 4 (same class!): Road={unsafe_road[4]:.3f} vs Nature={unsafe_nature[4]:.3f}")
print(f"  Delta at frame 4: {abs(unsafe_road[4]-unsafe_nature[4]):.3f} (should be large if M works)")

# V_acc comparison at frog frames
print(f"\n  V_acc at Frame 4 (first frog frame):")
print(f"    Road context:   pain={vacc_road[4][0]:.4f} pleasure={vacc_road[4][1]:.4f} epistemic={vacc_road[4][2]:.4f}")
print(f"    Nature context: pain={vacc_nature[4][0]:.4f} pleasure={vacc_nature[4][1]:.4f} epistemic={vacc_nature[4][2]:.4f}")
print(f"    = M provides DIFFERENT accumulated context -> classifier gets DIFFERENT signal")




  Temporal EEG: Context-Dependent (Same Frog, Different Meaning)
  Road+Frog UNSAFE probs:   ['0.000', '0.000', '0.000', '0.000', '1.000', '1.000', '1.000', '1.000']
  Nature+Frog UNSAFE probs: ['0.000', '0.000', '0.000', '0.000', '0.000', '0.000', '0.000', '0.000']
  Frog frame 4 (same class!): Road=1.000 vs Nature=0.000
  Delta at frame 4: 1.000 (should be large if M works)

  V_acc at Frame 4 (first frog frame):
    Road context:   pain=2.9295 pleasure=2.7565 epistemic=2.4945
    Nature context: pain=0.7372 pleasure=0.7050 epistemic=0.3129
    = M provides DIFFERENT accumulated context -> classifier gets DIFFERENT signal


In [12]:
# ════════════════════════════════════════════
# CELL 9: Module E+M Deep Analysis
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  Module E+M: Deep Analysis — Did They LEARN to Matter?")
print("=" * 90)

precs = adapter.get_precision().detach().cpu().numpy()
gammas = adapter.get_gamma().detach().cpu().numpy()

print(f"\n  Module E — Learned Precision Pi_l:")
for l in range(N_LAYERS):
    zone = "Perception" if l < 4 else "Evaluation" if l < 8 else "Decision"
    print(f"    Layer {l:2d} ({zone:10s}): Pi = {precs[l]:.4f}")

print(f"\n  Module M — Learned Decay Rates:")
print(f"    gamma_pain     = {gammas[0]:.4f} (slowest = trauma persists)")
print(f"    gamma_pleasure = {gammas[1]:.4f}")
print(f"    gamma_epistemic= {gammas[2]:.4f} (fastest = surprise resolves)")
print(f"    Ordering gamma_p < gamma_pl < gamma_e: {'CORRECT' if gammas[0] < gammas[1] < gammas[2] else 'WRONG'}")

# Compare across versions
print(f"\n  Comparison Across Versions:")
print(f"    {'Metric':<30s} {'LLM v6.1':>12s} {'Vision v2':>12s} {'Vision v3':>12s}")
print(f"    {'─'*66}")
print(f"    {'Pi_max':<30s} {'1.0070':>12s} {'~3.04':>12s} {precs.max():>12.4f}")
print(f"    {'Pi movement (Pi_max - 1.0)':<30s} {'0.0070':>12s} {'2.04':>12s} {precs.max()-1.0:>12.4f}")
print(f"    {'Ablation impact (overall)':<30s} {'0.1pp':>12s} {'0.0pp':>12s} {(res_full['overall']-res_no_EM['overall'])*100:>11.1f}pp")

drift_full = res_full.get('gradual_drift', 0)
drift_noem = res_no_EM.get('gradual_drift', 0)
ctx_full = (res_full.get('context_road_frog', 0) + res_full.get('context_nature_frog', 0)) / 2
ctx_noem = (res_no_EM.get('context_road_frog', 0) + res_no_EM.get('context_nature_frog', 0)) / 2
print(f"    {'Ablation: gradual drift':<30s} {'N/A':>12s} {'N/A':>12s} {(drift_full-drift_noem)*100:>11.1f}pp")
print(f"    {'Ablation: context-dependent':<30s} {'N/A':>12s} {'N/A':>12s} {(ctx_full-ctx_noem)*100:>11.1f}pp")

# Precision profile plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(range(N_LAYERS), precs, color=['#3498db']*4 + ['#2ecc71']*4 + ['#e74c3c']*4, edgecolor='black')
ax1.axhline(1.0, color='gray', ls='--', label='neutral (Pi=1)')
ax1.set_xlabel('Layer'); ax1.set_ylabel('Precision Pi_l')
ax1.set_title('Module E: Learned Precision Profile', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.bar(['gamma_pain', 'gamma_pleasure', 'gamma_epistemic'], gammas,
        color=['#e74c3c', '#2ecc71', '#3498db'], edgecolor='black')
ax2.set_ylabel('Decay Rate gamma')
ax2.set_title('Module M: Learned Decay Rates\n(gamma_p < gamma_pl < gamma_e)', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.suptitle('PBT-V v3: Learned E+M Parameters — Ninthanawat N.', fontweight='bold')
plt.tight_layout(); plt.savefig('pbtv3_em_analysis.png', dpi=150); plt.show()




  Module E+M: Deep Analysis — Did They LEARN to Matter?

  Module E — Learned Precision Pi_l:
    Layer  0 (Perception): Pi = 21.2378
    Layer  1 (Perception): Pi = 59.9913
    Layer  2 (Perception): Pi = 28.5754
    Layer  3 (Perception): Pi = 21.9866
    Layer  4 (Evaluation): Pi = 26.1701
    Layer  5 (Evaluation): Pi = 33.2864
    Layer  6 (Evaluation): Pi = 30.4715
    Layer  7 (Evaluation): Pi = 9.4298
    Layer  8 (Decision  ): Pi = 4.9150
    Layer  9 (Decision  ): Pi = 3.2635
    Layer 10 (Decision  ): Pi = 1.8628
    Layer 11 (Decision  ): Pi = 1.1134

  Module M — Learned Decay Rates:
    gamma_pain     = 0.0003 (slowest = trauma persists)
    gamma_pleasure = 0.0009
    gamma_epistemic= 0.0009 (fastest = surprise resolves)
    Ordering gamma_p < gamma_pl < gamma_e: CORRECT

  Comparison Across Versions:
    Metric                             LLM v6.1    Vision v2    Vision v3
    ──────────────────────────────────────────────────────────────────
    Pi_max                

In [13]:
# ════════════════════════════════════════════
# CELL 10: Temporal EEG — Tricky-Safe Controls
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  Control Tests")
print("=" * 90)

# Normal sequence — should stay calm
normal_idx = next(i for i, t in enumerate(test_types) if t == "normal_road")
adapter.reset_state()
unsafe_normal = []
with torch.no_grad():
    for t in range(SEQ_LEN):
        h_t = test_h[normal_idx][t].unsqueeze(0).to(device)
        logits, _, _ = adapter(h_t)
        unsafe_normal.append(F.softmax(logits, dim=-1)[0, 1].item())

# Drift-safe — should stay calm (drift between normal classes)
drift_safe_idx = next((i for i, t in enumerate(test_types) if t == "drift_safe"), None)
if drift_safe_idx is not None:
    adapter.reset_state()
    unsafe_drift_safe = []
    with torch.no_grad():
        for t in range(SEQ_LEN):
            h_t = test_h[drift_safe_idx][t].unsqueeze(0).to(device)
            logits, _, _ = adapter(h_t)
            unsafe_drift_safe.append(F.softmax(logits, dim=-1)[0, 1].item())
else:
    unsafe_drift_safe = [0]*SEQ_LEN

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(frames, unsafe_normal, 'g-o', lw=2, label='Normal road (should be calm)')
ax.plot(frames, unsafe_drift_safe, 'b-s', lw=2, label='Drift safe: car->truck (should be calm)')
ax.plot(frames, unsafe_drift, 'r-^', lw=2, label='Drift anomaly: car->airplane (should alarm!)')
ax.plot(frames, unsafe_road, 'm-x', lw=2, label='Road+frog (should alarm at frog!)')
ax.plot(frames, unsafe_nature, 'c-d', lw=2, label='Nature+frog (should stay calm)')
ax.axhline(0.5, color='gray', ls='--', alpha=0.5, label='Decision threshold')
ax.set_xticks(frames); ax.set_xlabel('Frame'); ax.set_ylabel('UNSAFE Probability')
ax.legend(loc='upper left', fontsize=9); ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
ax.set_title('PBT-V v3: All Sequence Types — UNSAFE Probability Over Time\nNinthanawat N.', fontweight='bold')
plt.tight_layout(); plt.savefig('pbtv3_all_types_comparison.png', dpi=150); plt.show()

over_refusal_normal = sum(1 for u in unsafe_normal if u > 0.5)
over_refusal_drift_safe = sum(1 for u in unsafe_drift_safe if u > 0.5)
print(f"  Normal road over-refusal: {over_refusal_normal}/{SEQ_LEN} (should be 0)")
print(f"  Drift-safe over-refusal: {over_refusal_drift_safe}/{SEQ_LEN} (should be 0)")




  Control Tests
  Normal road over-refusal: 0/8 (should be 0)
  Drift-safe over-refusal: 0/8 (should be 0)


In [14]:
# ════════════════════════════════════════════
# CELL 11: Final Summary
# ════════════════════════════════════════════
print("\n" + "=" * 90)
print("  FINAL SUMMARY — PBT-V v3: Making E+M Necessary")
print("=" * 90)

drift_impact_pp = (res_full.get("gradual_drift",0) - res_no_EM.get("gradual_drift",0)) * 100
ctx_road_impact_pp = (res_full.get("context_road_frog",0) - res_no_EM.get("context_road_frog",0)) * 100
ctx_nature_impact_pp = (res_full.get("context_nature_frog",0) - res_no_EM.get("context_nature_frog",0)) * 100
sudden_impact_pp = (res_full.get("sudden_anomaly",0) - res_no_EM.get("sudden_anomaly",0)) * 100
overall_impact_pp = (res_full.get("overall",0) - res_no_EM.get("overall",0)) * 100

em_necessary = drift_impact_pp > 2.0 or ctx_road_impact_pp > 2.0

print(f"""
+-----------------------------------------------------------------------+
| PBT-V v3: Making E+M NECESSARY                                        |
| DINOv2-Base (768 dim, 12 layers)                                       |
| CIFAR-10 Enhanced Temporal Sequences                                   |
| Predictive Boundary Theory -- Ninthanawat N.                           |
+-----------------------------------------------------------------------+
| Dataset: {n_total} sequences x {SEQ_LEN} frames = {n_total*SEQ_LEN:,} total frames              |
| Train: {len(train_h)} | Val: {len(val_h)} | Test: {len(test_h)} sequences                     |
+-----------------------------------------------------------------------+
| TASK TYPES:                                                            |
|   Normal road:         {N_NORMAL_ROAD} seq (control)                           |
|   Normal nature:       {N_NORMAL_NATURE} seq (control)                           |
|   Sudden anomaly:      {N_SUDDEN} seq (V can handle)                        |
|   Gradual drift:       {N_DRIFT_ANOM} seq (E+M critical!)                       |
|   Drift safe:          {N_DRIFT_SAFE} seq (drift control)                       |
|   Context road+frog:   {N_CTX_ROAD} seq (M critical!)                         |
|   Context nature+frog: {N_CTX_NATURE} seq (M critical!)                         |
+-----------------------------------------------------------------------+
| OVERALL ACCURACY:                                                      |
|   Full model (E+V+M): {res_full['overall']:.1%}                                         |
|   V only (no E+M):    {res_no_EM['overall']:.1%}                                         |
|   E+M impact:         {overall_impact_pp:+.1f}pp                                        |
+-----------------------------------------------------------------------+
| ABLATION BY TASK TYPE:                                                 |
|   Sudden anomaly:  Full={res_full.get('sudden_anomaly',0):.1%}  V-only={res_no_EM.get('sudden_anomaly',0):.1%}  impact={sudden_impact_pp:+.1f}pp       |
|   Gradual drift:   Full={res_full.get('gradual_drift',0):.1%}  V-only={res_no_EM.get('gradual_drift',0):.1%}  impact={drift_impact_pp:+.1f}pp       |
|   Road+frog:       Full={res_full.get('context_road_frog',0):.1%}  V-only={res_no_EM.get('context_road_frog',0):.1%}  impact={ctx_road_impact_pp:+.1f}pp       |
|   Nature+frog:     Full={res_full.get('context_nature_frog',0):.1%}  V-only={res_no_EM.get('context_nature_frog',0):.1%}  impact={ctx_nature_impact_pp:+.1f}pp       |
+-----------------------------------------------------------------------+
| Module E: Pi_max = {precs.max():.4f}  Pi_range = [{precs.min():.2f} - {precs.max():.2f}]                   |
| Module M: gamma_p={gammas[0]:.4f}  gamma_pl={gammas[1]:.4f}  gamma_e={gammas[2]:.4f}          |
| gamma ordering: {'gamma_p < gamma_pl < gamma_e CORRECT' if gammas[0]<gammas[1]<gammas[2] else 'WRONG'}                            |
+-----------------------------------------------------------------------+
| CONCLUSION: E+M {'ARE' if em_necessary else 'may not be'} NECESSARY for temporal/contextual tasks       |
|                                                                        |
|   V alone handles sudden changes (large epsilon per step)              |
|   E+M required for:                                                    |
|     1. Gradual drift (small epsilon per step, large cumulative)        |
|     2. Context-dependent (same object, different meaning by history)   |
|                                                                        |
|   = PBT HYPOTHESIS CONFIRMED:                                          |
|     "Module E+M activate when temporal/contextual processing           |
|      is NECESSARY, not just when temporal data is present"             |
+-----------------------------------------------------------------------+
""")

# Download
try:
    from google.colab import files
    files.download("pbt_vision_v3.pth")
    print("  Weights downloaded!")
except:
    print("Saved: pbt_vision_v3.pth")



  FINAL SUMMARY — PBT-V v3: Making E+M Necessary

+-----------------------------------------------------------------------+
| PBT-V v3: Making E+M NECESSARY                                        |
| DINOv2-Base (768 dim, 12 layers)                                       |
| CIFAR-10 Enhanced Temporal Sequences                                   |
| Predictive Boundary Theory -- Ninthanawat N.                           |
+-----------------------------------------------------------------------+
| Dataset: 1550 sequences x 8 frames = 12,400 total frames              |
| Train: 1085 | Val: 232 | Test: 233 sequences                     |
+-----------------------------------------------------------------------+
| TASK TYPES:                                                            |
|   Normal road:         300 seq (control)                           |
|   Normal nature:       200 seq (control)                           |
|   Sudden anomaly:      200 seq (V can handle)                     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Weights downloaded!
